# ResNet50 + Contrastive Regularization (train_from_scratch)
This notebook fine-tunes ResNet50 with a contrastive regularizer.

Sections:
- 1) Setup and Imports
- 2) Dataset and Dataloaders
- 3) Model and Loss
- 4) Training Loop
- 5) Evaluation and Plots

Tips: Set `lambda_contrastive=0.0` to run CE-only baselines.


In [1]:
# 1) Setup and Imports
# Basic utilities and HDF5 visualization helper
import h5py
from dl_utils.utils.utils import viz_h5_structure

In [2]:
with h5py.File('../../datasets/imagenet_v5_rot_10m_fix_vector-symmetry_vector.h5', 'r') as f:
    viz_h5_structure(f)

'Group': imagenet
  'Dataset': labels; Shape: (10173208,); dtype: uint8
  'Dataset': normalized_primitive_uc_vector_a; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_primitive_uc_vector_b; Shape: (10173208, 2); dtype: float32
  'Dataset': normalized_translation_start_point; Shape: (10173208, 2); dtype: float32
  'Dataset': primitive_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': primitive_uc_vector_b; Shape: (10173208, 2); dtype: int32
  'Dataset': rotation_angle; Shape: (10173208,); dtype: uint8
  'Dataset': shape; Shape: (10173208,); dtype: uint8
  'Dataset': similarity_vector; Shape: (10173208, 6); dtype: float32
  'Dataset': symmetry_vector; Shape: (10173208, 22); dtype: uint8
  'Dataset': translation_start_point; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_a; Shape: (10173208, 2); dtype: int32
  'Dataset': translation_uc_vector_b; Shape: (10173208, 2); dtype: int32


In [3]:
# Additional imports for training and visualization
# Model builder, trainer, and the contrastive-regularized loss wrapper
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from dl_utils.utils.dataset import hdf5_dataset, split_train_valid, viz_dataloader
from dl_utils.utils.utils import list_to_dict
from dl_utils.training.build_model import resnet50_
from dl_utils.training.trainer import Trainer, accuracy
from dl_utils.training.contrastive_loss import ContrastiveRegularizedLoss
import wandb

# Symmetry classes and helper to map label -> index
symmetry_classes = ['p1', 'p2', 'pm', 'pg', 'cm', 'pmm', 'pmg', 'pgg', 'cmm', 'p4', 'p4m', 'p4g', 'p3', 'p3m1', 'p31m', 'p6', 'p6m']
label_converter = list_to_dict(symmetry_classes)
n_classes = len(symmetry_classes)


/mnt/scratch/home/yichen/anaconda3/envs/symmetry/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 2) Config: dataset paths, training hyperparameters, and contrastive settings
task_name = "ResNet50-train_from_scratch-contrastive_reg-10m_v5"
ds_path_info = {
    'imagenet': '../../datasets/imagenet_v5_rot_10m_fix_vector.h5',
    'metadata': '../../datasets/imagenet_v5_rot_10m_fix_vector-symmetry_vector.h5',
    'atom': '../../datasets/atom_v5_rot_1m_fix_vector.h5',
    'noise': '../../datasets/noise_v5_rot_1m_fix_vector.h5',
    'viz_dataloader': False,  # set True to visualize few batches
}

training_specs = {
    'ds_size': 10_000_000,
    'batch_size': 3200,
    'num_workers': 12,
    'learning_rate': 1e-3,
    'training_image_count': 10_000_000,
    'validation_times': 100,
    'efficient_print': True,
    'model_path': '../../models/ResNet50/',
    'folder_name': 'default',
    'device_ids': [4,5,6,7,8,9],
    'epoch_start': 0,
    'contrastive_warmup_epochs': 10,
    'lambda_contrastive_start': 1e-20,
    'lambda_contrastive_final': 1e-5,
    'contrastive_lambda_schedule_type': 'log',
}

wandb_config = {
    'dataset': '10m datasets',
    'loss_func': 'CrossEntropyLoss',  # nn.MSELoss()
    'optimizer': 'Adam',
    'scheduler': 'OneCycleLR',
}
wandb_run_name = '10012025-' + task_name
proj_name = 'Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning'

# W&B (optional)
wandb_specs = {
    'project': proj_name,
    'entity': 'yig319',
    'group': 'contrastive',
    'save_code': True,
    'config': wandb_config,
    'resume': 'allow',
    'name': wandb_run_name,
    'id': wandb_run_name,
}

metadata_key = 'symmetry_vector'
metadata_weights = None
lambda_contrastive_start = training_specs.get('lambda_contrastive_start', 0.0)
lambda_contrastive_final = training_specs['lambda_contrastive_final']
contrastive_warmup_epochs = training_specs['contrastive_warmup_epochs']
contrastive_schedule_type = training_specs.get('contrastive_lambda_schedule_type', 'linear').lower()
metadata_distance = 'l2'
feature_layer = 'avgpool'
feature_norm = True


In [5]:
# 2b) Contrastive weight schedule

def contrastive_lambda_schedule(epoch_idx: int):
    """Warm up lambda_contrastive from start to final over the configured epochs."""
    if contrastive_warmup_epochs <= 0:
        return float(lambda_contrastive_final)

    clamped_epoch = max(0, epoch_idx)
    progress = min(1.0, clamped_epoch / contrastive_warmup_epochs)

    if contrastive_schedule_type == 'log':
        start = min(lambda_contrastive_start, 1e-5)
        end = max(lambda_contrastive_final, 1e-3)
        value = start * (end / start) ** progress
    else:
        value = lambda_contrastive_start + (lambda_contrastive_final - lambda_contrastive_start) * progress

    if progress >= 1.0:
        value = lambda_contrastive_final

    return float(value)

lambda_contrastive = contrastive_lambda_schedule(0)


In [6]:
# 2) Dataset and Dataloaders
# Imagenet with metadata for training
imagenet_all = hdf5_dataset(
    ds_path_info['imagenet'],
    folder='imagenet',
    transform=transforms.ToTensor(),
    metadata_keys=metadata_key,
    metadata_path=ds_path_info['metadata'],
    metadata_folder='imagenet',
)
ratio = training_specs['ds_size'] * (1 / 0.8) / len(imagenet_all)
imagenet_sub, _ = split_train_valid(imagenet_all, ratio, seed=42)
train_ds, valid_ds = split_train_valid(imagenet_sub, 0.8, seed=42)

train_dl = DataLoader(
    train_ds,
    batch_size=training_specs['batch_size'],
    shuffle=True,
    num_workers=training_specs['num_workers'],
)
valid_dl = DataLoader(
    valid_ds,
    batch_size=training_specs['batch_size'],
    shuffle=False,
    num_workers=training_specs['num_workers'],
)

# Noise CV
noise_ds_all = hdf5_dataset(ds_path_info['noise'], folder='noise', transform=transforms.ToTensor())
ratio_noise = np.min((training_specs['ds_size'] / len(noise_ds_all), 1))
noise_ds, _ = split_train_valid(noise_ds_all, ratio_noise, seed=42)
noise_dl = DataLoader(
    noise_ds,
    batch_size=training_specs['batch_size'],
    shuffle=False,
    num_workers=training_specs['num_workers'],
)

# Atom CV
atom_ds_all = hdf5_dataset(ds_path_info['atom'], folder='atom', transform=transforms.ToTensor())
ratio_atom = np.min((training_specs['ds_size'] / len(atom_ds_all), 1))
atom_ds, _ = split_train_valid(atom_ds_all, ratio_atom, seed=42)
atom_dl = DataLoader(
    atom_ds,
    batch_size=training_specs['batch_size'],
    shuffle=False,
    num_workers=training_specs['num_workers'],
)

if ds_path_info['viz_dataloader']:
    viz_dataloader(valid_dl, label_converter=label_converter, title='imagenet - valid')
    viz_dataloader(noise_dl, label_converter=label_converter, title='noise - valid')
    viz_dataloader(atom_dl, label_converter=label_converter, title='atom - valid')


idx = torch.randint(len(train_ds), (1,)).item()
img, lbl, meta = train_ds[idx]
print(lbl, symmetry_classes[lbl], meta['symmetry_vector'])

tensor(10, dtype=torch.uint8) p4m tensor([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0],
       dtype=torch.uint8)


In [7]:
# 3) Model: ResNet50 backbone (with optional DataParallel)
device = torch.device(f"cuda:{training_specs['device_ids'][0]}" if torch.cuda.is_available() else "cpu")

model = resnet50_(in_channels=3, n_classes=n_classes)
if len(training_specs['device_ids']) > 1:
    model = torch.nn.DataParallel(model, device_ids=training_specs['device_ids'])
model = model.to(device)

In [8]:
# 3b) Loss, optimizer, scheduler, and metrics
lr = training_specs['learning_rate']
epoch_start = training_specs.get('epoch_start', 0)
# epochs = training_specs['training_image_count'] // len(train_ds) - epoch_start
epochs = 100
valid_per_epochs = int(np.max((1, epochs / training_specs['validation_times'])))
early_stopping_patience = np.max((5, valid_per_epochs + 2))
efficient_print = training_specs['efficient_print']

base_ce = nn.CrossEntropyLoss()
loss_func = ContrastiveRegularizedLoss(
    base_criterion=base_ce,
    lambda_contrastive=lambda_contrastive,
    feature_layer=feature_layer,
    feature_norm=feature_norm,
    metadata_key=metadata_key,
    metadata_distance=metadata_distance,
    metadata_weights=metadata_weights,
)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    epochs=epochs,
    max_lr=lr,
    steps_per_epoch=len(train_dl),
)
metrics = [accuracy]

In [9]:
# 3c) Weights & Biases setup (optional)
NAME = task_name + '-dstaset_size=' + str(training_specs['ds_size'])
if wandb_specs:
    wandb.login()
    wandb.init(
        project=wandb_specs['project'], entity=wandb_specs['entity'],
        name=NAME, id=NAME, group=wandb_specs['group'],
        save_code=wandb_specs['save_code'], config=wandb_specs['config'], resume=wandb_specs['resume']
    )
    training_specs['wandb_record'] = True
else:
    training_specs['wandb_record'] = False

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: yig319 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# 4) Training Loop and Validation
Runs the Trainer and logs per-epoch metrics including base CE and contrastive components (if enabled).


In [10]:
folder_name = training_specs['folder_name'] if training_specs['folder_name'] != 'default' else NAME
model_dir = os.path.join(training_specs['model_path'], folder_name)
os.makedirs(model_dir, exist_ok=True)

trainer = Trainer(
    model=model,
    loss_func=loss_func,
    optimizer=optimizer,
    metrics=metrics,
    scheduler=scheduler,
    device=device,
    save_per_epochs=valid_per_epochs,
    model_path=model_dir + '/',
    early_stopping_patience=early_stopping_patience,
    efficient_print=efficient_print,
    print_batch_metrics=True,
)

history = trainer.train(
    train_dl=train_dl,
    epochs=epochs,
    epoch_start=epoch_start,
    valid_per_epochs=valid_per_epochs,
    valid_dl_list=[valid_dl, noise_dl, atom_dl],
    valid_dl_names=['', 'noise', 'atom'],
    wandb_record=training_specs['wandb_record'],
    contrastive_lambda_schedule=contrastive_lambda_schedule,
)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Epoch: 1/100
Train batch 1/2544 - loss: 3.1421, accuracy: 0.0556, loss_base: 3.1421, loss_contrastive: 0.1613
Train batch 2/2544 - loss: 2.8332, accuracy: 0.0622, loss_base: 2.8332, loss_contrastive: 0.0000
Train batch 3/2544 - loss: 2.8332, accuracy: 0.0612, loss_base: 2.8332, loss_contrastive: 0.0000
Train batch 4/2544 - loss: 2.8332, accuracy: 0.0572, loss_base: 2.8332, loss_contrastive: 0.0000
Train batch 5/2544 - loss: 2.8332, accuracy: 0.0566, loss_base: 2.8332, loss_contrastive: 0.0000
Train batch 6/2544 - loss: 2.8332, accuracy: 0.0531, loss_base: 2.8332, loss_contrastive: 0.0000
Train batch 7/

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f237cc8e110>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f24404dce90, execution_count=10 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f2501ad93d0, raw_cell="folder_name = training_specs['folder_name'] if tra.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a2241313030227d/scratch/home/yichen/Understanding-Experimental-Images-by-Identifying-Symmetries-with-Deep-Learning/notebooks/CRISPS/benchmark-v5-resnet50-contrastive_reg-train_from_scratchv2.ipynb#X14sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


MailboxClosedError: 